In [219]:
"""
from pathlib import Path
import subprocess
import sys

ROOT = Path.cwd()
VENV_DIR = ROOT / ".venv"
PYTHON_BIN = VENV_DIR / "bin" / "python"

if not VENV_DIR.exists():
    subprocess.run([sys.executable, "-m", "venv", str(VENV_DIR)], check=True)

subprocess.run([str(PYTHON_BIN), "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([str(PYTHON_BIN), "-m", "pip", "install", "jupyter", "ipykernel", "pandas", "pyarrow", "scikit-learn", "xgboost"], check=True)
subprocess.run([str(PYTHON_BIN), "-m", "ipykernel", "install", "--user", "--name", "songgraph-xgboost", "--display-name", "SongGraph XGBoost"], check=True)

print(VENV_DIR)
"""

'\nfrom pathlib import Path\nimport subprocess\nimport sys\n\nROOT = Path.cwd()\nVENV_DIR = ROOT / ".venv"\nPYTHON_BIN = VENV_DIR / "bin" / "python"\n\nif not VENV_DIR.exists():\n    subprocess.run([sys.executable, "-m", "venv", str(VENV_DIR)], check=True)\n\nsubprocess.run([str(PYTHON_BIN), "-m", "pip", "install", "--upgrade", "pip"], check=True)\nsubprocess.run([str(PYTHON_BIN), "-m", "pip", "install", "jupyter", "ipykernel", "pandas", "pyarrow", "scikit-learn", "xgboost"], check=True)\nsubprocess.run([str(PYTHON_BIN), "-m", "ipykernel", "install", "--user", "--name", "songgraph-xgboost", "--display-name", "SongGraph XGBoost"], check=True)\n\nprint(VENV_DIR)\n'

In [235]:
from pathlib import Path
import subprocess

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_validate, train_test_split
from xgboost import XGBRegressor

ROOT = Path.cwd()
DATA_PATH = ROOT / "liveness_final.parquet"
#DATA_PATH = ROOT / "energy_final.parquet"

build_info = xgb.build_info()
cuda_built = str(build_info.get("USE_CUDA", "OFF")).upper() in {"ON", "TRUE", "1"}

try:
    nvidia_smi = subprocess.run(
        ["nvidia-smi"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=False,
    )
    cuda_visible = nvidia_smi.returncode == 0
except FileNotFoundError:
    cuda_visible = False

XGB_DEVICE = "cuda" if cuda_built and cuda_visible else "cpu"
XGB_DEVICE


'cpu'

In [221]:
TARGET_COLUMNS = [
    "valence",
    "loudness",
    "danceability",
    "energy",
    "acousticness",
    "instrumentalness",
    "liveness",
    "speechiness",
    "key",
    "mode",
]

FEATURE_POOL = [
    "valence",
    "loudness",
    "danceability",
    "energy",
    "tempo",
    "acousticness",
    "instrumentalness",
    "liveness",
    "speechiness",
    "key",
    "mode",
    "valence_missing",
    "loudness_missing",
    "danceability_missing",
    "energy_missing",
    "tempo_missing",
    "acousticness_missing",
    "instrumentalness_missing",
    "liveness_missing",
    "speechiness_missing",
    "key_missing",
    "mode_missing",
]

TARGET = "liveness"
feature_columns = [column for column in FEATURE_POOL if column != TARGET]
OUTPUT_PATH = ROOT / f"{TARGET}_final.parquet"

MODEL_PARAMS = {
    "objective": "reg:squarederror",
    "n_estimators": 600,
    "max_depth": 6,
    "learning_rate": 0.03,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 1.0,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
    "gamma": 0.0,
    "tree_method": "hist",
    "device": XGB_DEVICE,
    "random_state": 42,
    "n_jobs": -1,
}

TARGET


'liveness'

In [236]:
columns_to_load = sorted(set(FEATURE_POOL + ["track_id", "artist_name", "track_name"]))
df = pd.read_parquet(DATA_PATH, columns=columns_to_load)
float64_columns = df.select_dtypes(include=["float64"]).columns
df[float64_columns] = df[float64_columns].astype("float32")
df.shape


(3374899, 25)

In [237]:
df[TARGET_COLUMNS].isna().sum().sort_values(ascending=False)


key                 446161
valence                  0
loudness                 0
danceability             0
energy                   0
acousticness             0
instrumentalness         0
liveness                 0
speechiness              0
mode                     0
dtype: int64

In [224]:
observed_df = df[df[TARGET].notna()].copy()
missing_df = df[df[TARGET].isna()].copy()

X_observed = observed_df[feature_columns]
y_observed = observed_df[TARGET]
X_missing = missing_df[feature_columns]

X_train, X_test, y_train, y_test = train_test_split(
    X_observed,
    y_observed,
    test_size=0.2,
    random_state=42,
)

print(TARGET)
print(len(feature_columns))
print(len(observed_df))
print(len(missing_df))
print(len(X_train))
print(len(X_test))


liveness
21
3374899
0
2699919
674980


In [225]:
model = XGBRegressor(**MODEL_PARAMS)

cv = KFold(n_splits=5, shuffle=True, random_state=42)

cv_results = cross_validate(
    estimator=model,
    X=X_train,
    y=y_train,
    cv=cv,
    scoring={
        "rmse": "neg_root_mean_squared_error",
        "mae": "neg_mean_absolute_error",
        "r2": "r2",
    },
    n_jobs=-1,
    return_train_score=False,
)

cv_metrics = {
    "val_rmse": -cv_results["test_rmse"].mean(),
    "val_mae": -cv_results["test_mae"].mean(),
    "val_r2": cv_results["test_r2"].mean(),
}

cv_metrics


{'val_rmse': np.float64(0.183150315284729),
 'val_mae': np.float64(0.1338093250989914),
 'val_r2': np.float64(0.15292510986328126)}

In [226]:
model = XGBRegressor(**MODEL_PARAMS)
model.fit(X_train, y_train)

predictions = model.predict(X_test).astype(np.float32)
predictions = np.clip(predictions, 0, 1)

test_metrics = {
    "mae": mean_absolute_error(y_test, predictions),
    "rmse": np.sqrt(mean_squared_error(y_test, predictions)),
    "r2": r2_score(y_test, predictions),
}

test_metrics


{'mae': 0.13393107056617737,
 'rmse': np.float64(0.18330482311802238),
 'r2': 0.15339982509613037}

In [227]:
final_model = XGBRegressor(**MODEL_PARAMS)
final_model.fit(X_observed, y_observed)

predictions = final_model.predict(X_missing).astype(np.float32)
predictions = np.clip(predictions, 0, 1)

imputed_rows = missing_df[["track_id", "artist_name", "track_name"]].copy()
imputed_rows[TARGET] = predictions

imputed_rows.head()


,track_id,artist_name,track_name,liveness


In [228]:
print(imputed_rows[TARGET].describe())
print("---")
print(imputed_rows[TARGET].quantile([0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))
print("---")
print(
imputed_rows[TARGET].min(),
imputed_rows[TARGET].max(),
imputed_rows[TARGET].mean(),
imputed_rows[TARGET].std(),
)
print("---")
print(imputed_rows[TARGET].nunique())

count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: liveness, dtype: float64
---
0.01   NaN
0.05   NaN
0.25   NaN
0.50   NaN
0.75   NaN
0.95   NaN
0.99   NaN
Name: liveness, dtype: float64
---
nan nan nan nan
---
0


In [229]:
completed_rows = df[df[TARGET].notna()].copy()

print(completed_rows[TARGET].describe())
print("---")
print(completed_rows[TARGET].quantile([0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))
print("---")
print(
completed_rows[TARGET].min(),
completed_rows[TARGET].max(),
completed_rows[TARGET].mean(),
completed_rows[TARGET].std(),
)
print("---")
print(completed_rows[TARGET].nunique())

count    3.374899e+06
mean     3.387391e-01
std      1.990421e-01
min      0.000000e+00
25%      1.956522e-01
50%      2.719486e-01
75%      4.328377e-01
max      1.000000e+00
Name: liveness, dtype: float64
---
0.01    0.076874
0.05    0.121441
0.25    0.195652
0.50    0.271949
0.75    0.432838
0.95    0.764454
0.99    0.929674
Name: liveness, dtype: float64
---
0.0 1.0 0.3387391 0.1990421
---
334562


In [230]:
missing_idx = df.index[df[TARGET].isna()]
df.loc[missing_idx, TARGET] = predictions
df[[TARGET]].isna().sum()


liveness    0
dtype: int64

In [231]:
df.to_parquet(OUTPUT_PATH)
OUTPUT_PATH


PosixPath('/Users/saake/SongGraph/data_cleaning/liveness_final.parquet')